# RIME-X Steps 2 & 3: Warming Levels + Quantile Maps + Emulator
Built from the actual rimeX source code (`quantilemaps.py`).

**Pipeline:**
```
Your GMT CSVs (all 4 models)
        ↓
Step 1: Compute warming levels (year each model hits 1.0°C, 1.1°C ... 4.0°C)
        ↓
Step 2: Build quantile maps  (make_quantile_map_array)
        ↓
Step 3: Run emulator with MAGICC  (make_quantilemap_prediction)
        ↓
Probabilistic regional projections (plots + CSVs)
```


## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import os, glob
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Imports done ✅")


## Cell 2 — Paths Configuration

In [ ]:
# ── UPDATE THESE PATHS ───────────────────────────────────────────────
ROOT        = '/media/wcs/Disk3/imran/GCM_models'
OUTPUT_ROOT = f'{ROOT}/all_models_regional_output'
MAGICC_CSV  = f'{ROOT}/ssp245_MAGICC_separated.csv'
RIMEX_OUT   = f'{ROOT}/rimex_results'

MODELS = ['CNRM-ESM2-1', 'IPSL-CM6A-LR', 'MPI-ESM1-2-LR', 'UKESM1-0-LL']
INDICATOR = 'tas'

# Pre-industrial baseline (standard CMIP6 definition)
PREINDUSTRIAL_BASELINE = (1850, 1900)

# Warming level steps to compute
WL_MIN  = 1.0
WL_MAX  = 4.0
WL_STEP = 0.1
WARMING_LEVELS = np.round(np.arange(WL_MIN, WL_MAX + WL_STEP, WL_STEP), 1)

# Running mean window (years) — RIME-X default is 21
RUNNING_MEAN_WINDOW = 21

# Number of quantile bins — RIME-X default is 101
QUANTILE_BINS = 101

os.makedirs(RIMEX_OUT, exist_ok=True)

print(f"ROOT          : {ROOT}")
print(f"MAGICC CSV    : {MAGICC_CSV} exists={os.path.exists(MAGICC_CSV)}")
print(f"Warming levels: {WARMING_LEVELS[0]}°C to {WARMING_LEVELS[-1]}°C "
      f"({len(WARMING_LEVELS)} levels)")
print(f"Quantile bins : {QUANTILE_BINS}")


## Cell 3 — Load GMT CSVs from All Models

In [ ]:
# Collect all global mean temperature CSVs from all 4 models

gmt_records = []   # list of dicts: {model, scenario, ensemble, df}

for model in MODELS:
    gm_dir = f'{OUTPUT_ROOT}/{model}/cmip-6_global_mean/{INDICATOR}'
    if not os.path.exists(gm_dir):
        print(f"  ⚠️  {model}: GMT dir not found — {gm_dir}")
        continue

    csv_files = glob.glob(f'{gm_dir}/globalmean_*.csv')
    for fpath in csv_files:
        fname  = os.path.basename(fpath)
        # filename: globalmean_tas_cnrm-esm2-1_ssp245-r1i1p1f2.csv
        parts  = fname.replace('.csv','').split('_')
        # parts: ['globalmean','tas','cnrm-esm2-1','ssp245-r1i1p1f2']
        sc_ens = parts[3]
        scen   = sc_ens.split('-')[0]          # ssp245
        ens    = '-'.join(sc_ens.split('-')[1:]) # r1i1p1f2

        df = pd.read_csv(fpath)
        # Convert time to year integer
        df['year'] = pd.to_datetime(df['time']).dt.year
        # Annual mean from monthly data
        df_annual  = df.groupby('year')['tas'].mean().reset_index()

        gmt_records.append({
            'model'    : model,
            'scenario' : scen,
            'ensemble' : ens,
            'df'       : df_annual,
        })
        print(f"  {model:20s} | {scen:12s} | {ens:15s} | "
              f"{df_annual['year'].min()}–{df_annual['year'].max()} "
              f"({len(df_annual)} years)")

print(f"\nTotal GMT series loaded: {len(gmt_records)}")


## Cell 4 — Compute Warming Levels

In [ ]:
# For each model/scenario/ensemble:
#   1. Compute anomaly relative to pre-industrial baseline (1850-1900)
#   2. Apply 21-year centred running mean
#   3. Find the year when running mean first crosses each warming level

def compute_warming_levels(gmt_records, warming_levels,
                            baseline=PREINDUSTRIAL_BASELINE,
                            window=RUNNING_MEAN_WINDOW):

    records = []

    for rec in gmt_records:
        model    = rec['model']
        scenario = rec['scenario']
        ensemble = rec['ensemble']
        df       = rec['df'].copy()

        # Skip historical — warming levels computed from projections
        if scenario == 'historical':
            continue

        # Need historical data for baseline — find it
        hist_rec = next(
            (r for r in gmt_records
             if r['model'] == model
             and r['scenario'] == 'historical'
             and r['ensemble'] == ensemble),
            None
        )

        # Concatenate historical + projection
        if hist_rec is not None:
            df_full = pd.concat(
                [hist_rec['df'], df], ignore_index=True
            ).drop_duplicates('year').sort_values('year')
        else:
            df_full = df.sort_values('year')

        df_full = df_full.set_index('year')

        # Anomaly relative to pre-industrial baseline
        y1, y2  = baseline
        if y1 in df_full.index and y2 in df_full.index:
            baseline_mean = df_full.loc[y1:y2, 'tas'].mean()
        else:
            # Use earliest available years as fallback
            baseline_mean = df_full['tas'].iloc[:20].mean()

        df_full['anomaly'] = df_full['tas'] - baseline_mean

        # 21-year centred running mean (min_periods = window//2 to allow edges)
        df_full['smooth'] = df_full['anomaly'].rolling(
            window=window, center=True, min_periods=window//2
        ).mean()

        # Find first year each warming level is crossed
        for wl in warming_levels:
            crossed = df_full[df_full['smooth'] >= wl]
            if len(crossed) == 0:
                continue   # this model never reaches this WL
            year_crossed = int(crossed.index[0])
            records.append({
                'warming_level' : wl,
                'year'          : year_crossed,
                'model'         : model.lower(),
                'experiment'    : scenario,
                'ensemble'      : ensemble,
            })

    wl_df = pd.DataFrame(records).sort_values(
        ['model', 'experiment', 'warming_level']
    ).reset_index(drop=True)

    return wl_df


print("Computing warming levels...")
wl_df = compute_warming_levels(gmt_records, WARMING_LEVELS)

# Save
wl_path = f'{RIMEX_OUT}/warming_levels.csv'
wl_df.to_csv(wl_path, index=False)

print(f"\nWarming levels table: {wl_df.shape}")
print(f"Saved to: {wl_path}")
print(f"\nPreview:")
print(wl_df.head(12).to_string(index=False))
print(f"\nModels covered      : {wl_df['model'].unique()}")
print(f"Scenarios covered   : {wl_df['experiment'].unique()}")
print(f"WL range            : {wl_df['warming_level'].min()}°C "
      f"to {wl_df['warming_level'].max()}°C")


## Cell 5 — Index All Regional CSV Files

In [ ]:
# Build a lookup table: (model, scenario, ensemble, region) → csv_path
# This lets make_quantile_map_array find the right file for any combination

print("Indexing regional CSV files...")

file_index = {}   # key: (model_lower, scenario, region) → path

for model in MODELS:
    reg_dir = f'{OUTPUT_ROOT}/{model}/isimip_regional_data'
    if not os.path.exists(reg_dir):
        print(f"  ⚠️  {model}: regional dir not found")
        continue

    for region in os.listdir(reg_dir):
        lw_dir = f'{reg_dir}/{region}/latWeight'
        if not os.path.exists(lw_dir):
            continue
        for fname in os.listdir(lw_dir):
            if not fname.endswith('.csv'):
                continue
            parts    = fname.replace('.csv','').split('_')
            # parts: model, scenario-ensemble, indicator, region...
            # e.g.: cnrm-esm2-1_ssp245-r1i1p1f2_tas_pak_punjab_123_latweight
            model_l  = parts[0]
            sc_ens   = parts[1]
            scen     = sc_ens.split('-')[0]
            key = (model_l, scen, region)
            file_index[key] = f'{lw_dir}/{fname}'

print(f"Total indexed files : {len(file_index)}")

# Show unique regions available
all_regions = sorted(set(k[2] for k in file_index.keys()))
print(f"Unique regions      : {len(all_regions)}")
print(f"Sample regions      : {all_regions[:5]}")

# Show models/scenarios indexed
combos = sorted(set((k[0], k[1]) for k in file_index.keys()))
print(f"\nModel/scenario combinations ({len(combos)}):")
for m, s in combos:
    print(f"  {m:25s} | {s}")


## Cell 6 — Build Quantile Maps

In [ ]:
# Implements make_quantile_map_array logic from rimeX/preproc/quantilemaps.py
# For each region: collects all model/scenario slices at each warming level
# and computes 101 quantiles of the regional indicator distribution

def build_quantile_map_for_region(
    region,
    file_index,
    wl_df,
    quantile_bins=101,
    running_mean_window=21,
    equiprobable_models=True,
):
    """
    For one region, builds the quantile map:
      dims: warming_level × quantile
      value: regional indicator value at that (WL, quantile) combination
    """
    quants   = np.linspace(0, 1, quantile_bins)
    wl_vals  = sorted(wl_df['warming_level'].unique())
    w        = running_mean_window // 2

    # collect: {wl: {'values': [], 'models': []}}
    collect  = {wl: {'values': [], 'models': []} for wl in wl_vals}

    for (model_l, scen, reg), fpath in file_index.items():
        if reg != region:
            continue
        if scen == 'historical':
            continue

        # Find matching rows in warming level table
        wl_rows = wl_df[
            (wl_df['model'] == model_l) &
            (wl_df['experiment'] == scen)
        ]
        if len(wl_rows) == 0:
            continue

        # Load regional timeseries
        df_reg = pd.read_csv(fpath)
        col    = [c for c in df_reg.columns if c != 'time'][0]

        df_reg['year'] = pd.to_datetime(df_reg['time']).dt.year
        df_annual = df_reg.groupby('year')[col].mean()

        # Also load historical if available
        hist_key = (model_l, 'historical', region)
        if hist_key in file_index:
            df_hist = pd.read_csv(file_index[hist_key])
            df_hist['year'] = pd.to_datetime(df_hist['time']).dt.year
            df_hist_a = df_hist.groupby('year')[col].mean()
            combined  = pd.concat([df_hist_a, df_annual]).sort_index()
            combined  = combined[~combined.index.duplicated()]
        else:
            combined  = df_annual

        # 21-year running mean
        smooth = combined.rolling(
            window=running_mean_window, center=True, min_periods=w
        ).mean()

        # Collect the smoothed value at each warming level year
        for _, row in wl_rows.iterrows():
            wl   = row['warming_level']
            year = int(row['year'])
            if year in smooth.index and np.isfinite(smooth.loc[year]):
                collect[wl]['values'].append(float(smooth.loc[year]))
                collect[wl]['models'].append(model_l)

    # Compute quantiles at each warming level
    qmap_values = np.full((len(wl_vals), quantile_bins), np.nan)

    for i, wl in enumerate(wl_vals):
        values = np.array(collect[wl]['values'])
        models = collect[wl]['models']
        if len(values) < 2:
            continue

        if equiprobable_models:
            # Weight each sample inversely by how many times its model appears
            from collections import Counter
            freq    = Counter(models)
            weights = np.array([1.0 / freq[m] for m in models])
            weights /= weights.sum()
            # Weighted quantile
            sorted_idx = np.argsort(values)
            values_s   = values[sorted_idx]
            weights_s  = weights[sorted_idx]
            cum_w      = np.cumsum(weights_s) - weights_s / 2
            qmap_values[i] = np.interp(quants, cum_w, values_s)
        else:
            qmap_values[i] = np.quantile(values, quants)

    da = xr.DataArray(
        qmap_values,
        dims=['warming_level', 'quantile'],
        coords={
            'warming_level': wl_vals,
            'quantile'     : quants,
        },
        name=region,
    )
    return da


print("Function defined ✅")


In [ ]:
# ── BUILD QUANTILE MAPS FOR ALL REGIONS ──────────────────────────────
# Select which regions to build — start with PAK regions for speed,
# then change to all_regions to run for everything

# Option A: PAK regions only (fast, for testing)
target_regions = [r for r in all_regions if 'pak' in r.lower()]

# Option B: ALL regions (slow, full run — uncomment when ready)
# target_regions = all_regions

print(f"Building quantile maps for {len(target_regions)} regions...")
print(f"Sample: {target_regions[:5]}")

QM_DIR = f'{RIMEX_OUT}/quantile_maps'
os.makedirs(QM_DIR, exist_ok=True)

failed = []

for region in tqdm(target_regions, desc="Quantile maps"):
    out_path = f'{QM_DIR}/tas_annual_{region}_latweight_eq.nc'
    if os.path.exists(out_path):
        continue   # skip if already built

    try:
        da = build_quantile_map_for_region(
            region        = region,
            file_index    = file_index,
            wl_df         = wl_df,
            quantile_bins = QUANTILE_BINS,
            running_mean_window = RUNNING_MEAN_WINDOW,
        )
        # Save as NetCDF
        da.to_netcdf(out_path)

    except Exception as e:
        failed.append((region, str(e)))

built = [f for f in os.listdir(QM_DIR) if f.endswith('.nc')]
print(f"\nQuantile maps built : {len(built)}")
print(f"Failed              : {len(failed)}")
if failed:
    for r, e in failed[:5]:
        print(f"  {r}: {e}")


## Cell 7 — Preview a Quantile Map

In [ ]:
# Load and inspect one quantile map
qm_files = sorted([f for f in os.listdir(QM_DIR) if f.endswith('.nc')])

if qm_files:
    sample_path = f'{QM_DIR}/{qm_files[0]}'
    da = xr.open_dataarray(sample_path)
    region_name = qm_files[0].split('_latweight')[0].replace('tas_annual_','')

    print(f"Quantile map: {qm_files[0]}")
    print(f"Shape       : {da.shape}")
    print(f"Dims        : {da.dims}")
    print(f"WL range    : {float(da.warming_level.min()):.1f}°C "
          f"to {float(da.warming_level.max()):.1f}°C")
    print(f"Quantiles   : {da.quantile.values[:5]}...{da.quantile.values[-5:]}")
    print(f"\nValues at 1.5°C warming (selected quantiles):")
    wl15 = da.sel(warming_level=1.5, method='nearest')
    for q in [0.05, 0.17, 0.50, 0.83, 0.95]:
        val = float(wl15.sel(quantile=q, method='nearest').values)
        print(f"  p{int(q*100):2d} = {val:.3f}°C")

    # Plot the quantile map
    fig, ax = plt.subplots(figsize=(10, 5))
    for q, color, label in [
        (0.05,  'steelblue', 'p5'),
        (0.17,  'cornflowerblue', 'p17'),
        (0.50,  'navy',     'median'),
        (0.83,  'orange',   'p83'),
        (0.95,  'red',      'p95'),
    ]:
        vals = da.sel(quantile=q, method='nearest').values
        ax.plot(da.warming_level.values, vals, label=label,
                linewidth=2 if q == 0.5 else 1)

    ax.set_xlabel("Global Warming Level (°C above pre-industrial)", fontsize=12)
    ax.set_ylabel("Regional tas anomaly (°C)", fontsize=12)
    ax.set_title(f"Quantile Map — {region_name} | tas | annual", fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{RIMEX_OUT}/quantilemap_{region_name}.png', dpi=150)
    plt.show()
    da.close()


## Cell 8 — Run Emulator with MAGICC 600 Ensembles

In [ ]:
# Implements make_quantilemap_prediction from rimeX/preproc/quantilemaps.py
# Combines 600 MAGICC GMT trajectories with the quantile map
# to produce 600 regional temperature trajectories

def make_quantilemap_prediction(
    quantile_map,    # xr.DataArray: dims (warming_level, quantile)
    gmt,             # pd.DataFrame: index=year, columns=ensemble members
    samples=5000,
    seed=42,
    quantiles=[0.05, 0.17, 0.50, 0.83, 0.95],
    clip=True,
):
    """
    Exact reimplementation of rimeX make_quantilemap_prediction.
    Returns xr.DataArray: dims (year, quantile)
    """
    rng = np.random.default_rng(seed=seed)

    wl_vals = quantile_map.warming_level.values
    q_vals  = quantile_map.quantile.values

    if clip:
        gmt = gmt.clip(lower=wl_vals[0], upper=wl_vals[-1])

    # Deterministic resampling of GMT distribution
    gmt_quants     = np.linspace(0, 1, samples)
    resampled_gmt  = np.quantile(gmt.values, gmt_quants, axis=1).T  # (years, samples)

    # Deterministic resampling of impact quantiles (shuffled)
    impact_quants  = np.linspace(0, 1, samples)
    rng.shuffle(impact_quants)

    # Build 2D interpolator: (warming_level, quantile) → regional_value
    interp = RegularGridInterpolator(
        (wl_vals, q_vals),
        quantile_map.values,
        bounds_error=False,
        fill_value=np.nan,
    )

    # For each year and each sample: look up regional value
    # resampled_gmt: (n_years, samples)
    # impact_quants: (samples,)
    n_years  = resampled_gmt.shape[0]
    points   = np.stack([
        resampled_gmt.ravel(),
        np.tile(impact_quants, n_years),
    ], axis=1)

    sampled  = interp(points).reshape(n_years, samples)  # (n_years, samples)

    # Compute output quantiles
    result   = np.quantile(sampled, quantiles, axis=1).T  # (n_years, n_quantiles)

    da = xr.DataArray(
        result,
        dims=['year', 'quantile'],
        coords={
            'year'    : gmt.index,
            'quantile': quantiles,
        },
    )
    return da


print("Emulator function defined ✅")


In [ ]:
# Load MAGICC 600-ensemble GMT file
gmt_magicc = pd.read_csv(MAGICC_CSV, index_col='year')
gmt_magicc.columns = gmt_magicc.columns.astype(str)

# Apply pre-industrial baseline correction
# (MAGICC already outputs anomalies, but verify with your file)
y1, y2 = PREINDUSTRIAL_BASELINE
if y1 in gmt_magicc.index:
    gmt_magicc -= gmt_magicc.loc[y1:y2].mean()

print(f"MAGICC GMT shape : {gmt_magicc.shape}")
print(f"Years            : {gmt_magicc.index.min()} to {gmt_magicc.index.max()}")
print(f"Ensembles        : {gmt_magicc.shape[1]}")
print(f"\n2050 GMT range   : "
      f"{gmt_magicc.loc[2050].min():.2f}°C to "
      f"{gmt_magicc.loc[2050].max():.2f}°C")
print(f"2100 GMT range   : "
      f"{gmt_magicc.loc[2100].min():.2f}°C to "
      f"{gmt_magicc.loc[2100].max():.2f}°C")


In [ ]:
# ── RUN EMULATOR FOR ALL BUILT REGIONS ───────────────────────────────

EMU_OUT = f'{RIMEX_OUT}/emulator_output'
os.makedirs(EMU_OUT, exist_ok=True)

OUTPUT_QUANTILES = [0.05, 0.17, 0.50, 0.83, 0.95]

qm_files_available = sorted([f for f in os.listdir(QM_DIR) if f.endswith('.nc')])
print(f"Running emulator for {len(qm_files_available)} regions...")

for qm_file in tqdm(qm_files_available, desc="Emulating"):
    region_name = qm_file.replace('tas_annual_','').replace('_latweight_eq.nc','')
    out_csv     = f'{EMU_OUT}/{region_name}_tas_ssp245_emulated.csv'

    if os.path.exists(out_csv):
        continue

    qm = xr.open_dataarray(f'{QM_DIR}/{qm_file}')

    # Check the map has valid data
    if not np.isfinite(qm.values).any():
        qm.close()
        continue

    result = make_quantilemap_prediction(
        quantile_map = qm,
        gmt          = gmt_magicc,
        samples      = 5000,
        seed         = 42,
        quantiles    = OUTPUT_QUANTILES,
        clip         = True,
    )
    qm.close()

    # Save CSV
    df_out = result.to_pandas()
    df_out.columns = [f'p{int(q*100)}' for q in OUTPUT_QUANTILES]
    df_out.index.name = 'year'
    df_out.to_csv(out_csv)

emu_files = os.listdir(EMU_OUT)
print(f"\nEmulator outputs: {len(emu_files)} files")
print(f"Saved to: {EMU_OUT}")


## Cell 9 — Plot Probabilistic Projections

In [ ]:
# Plot all emulated regions
emu_files = sorted([f for f in os.listdir(EMU_OUT) if f.endswith('.csv')])
n_regions_emulated = len(emu_files)

fig, axes = plt.subplots(
    max(1, (n_regions_emulated + 2) // 3),
    min(3, n_regions_emulated),
    figsize=(18, 5 * max(1, (n_regions_emulated + 2) // 3)),
    squeeze=False
)
axes_flat = axes.flatten()

for idx, fname in enumerate(emu_files):
    region_name = fname.replace('_tas_ssp245_emulated.csv','')
    df = pd.read_csv(f'{EMU_OUT}/{fname}', index_col='year')

    ax = axes_flat[idx]
    ax.fill_between(df.index, df['p5'],  df['p95'],
                    alpha=0.15, color='steelblue', label='5–95th %ile')
    ax.fill_between(df.index, df['p17'], df['p83'],
                    alpha=0.30, color='steelblue', label='17–83rd %ile')
    ax.plot(df.index, df['p50'],
            color='steelblue', linewidth=2, label='Median')

    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(region_name.upper().replace('_',' '), fontsize=10)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('tas anomaly (°C)', fontsize=8)
    ax.grid(True, alpha=0.3)

    if idx == 0:
        ax.legend(fontsize=8)

# Hide unused subplots
for idx in range(len(emu_files), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle(
    f'RIME-X Probabilistic tas Projections | SSP2-4.5 | 600 MAGICC ensembles\n'
    f'4 GCMs: CNRM-ESM2-1, IPSL-CM6A-LR, MPI-ESM1-2-LR, UKESM1-0-LL',
    fontsize=13, y=1.01
)
plt.tight_layout()
plot_path = f'{RIMEX_OUT}/all_regions_projections.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved: {plot_path}")


## Cell 10 — Final Summary

In [ ]:
print("=" * 60)
print("RIME-X PIPELINE COMPLETE")
print("=" * 60)

print(f"\n1. Warming levels   : {RIMEX_OUT}/warming_levels.csv")
wl_built = pd.read_csv(f'{RIMEX_OUT}/warming_levels.csv')
print(f"   Rows: {len(wl_built)} | WL: {wl_built.warming_level.min()}–"
      f"{wl_built.warming_level.max()}°C")

print(f"\n2. Quantile maps    : {QM_DIR}/")
qm_count = len([f for f in os.listdir(QM_DIR) if f.endswith('.nc')])
print(f"   Files: {qm_count} NetCDF files (one per region)")

print(f"\n3. Emulator outputs : {EMU_OUT}/")
emu_count = len([f for f in os.listdir(EMU_OUT) if f.endswith('.csv')])
print(f"   Files: {emu_count} CSV files (one per region)")

print(f"\n4. Plots            : {RIMEX_OUT}/")
print()
print("Output columns in each emulator CSV:")
print("  year | p5 | p17 | p50 | p83 | p95")
print()
print("These are °C anomalies relative to pre-industrial baseline")
print("at each warming level, driven by 600 MAGICC SSP2-4.5 ensembles")
print("across 4 CMIP6 models.")
print("=" * 60)
